[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tymworld/DROPPS/blob/v1.0.0/examples/colab/02_DROPPS_1_0_Simulation.ipynb)

# DROPPS 1.0 Phase-Coexistence Simulation

This notebook implements the slab protocol in its required order:

1. start from many protein monomers in a compact cubic box;
2. condense/equilibrate the box in the NPT ensemble;
3. expand the **z box axis tenfold**, recentering the condensed
   configuration in the resulting elongated box;
4. run the elongated box in the NVT ensemble;
5. pass only `slab_nvt.tpr` and `slab_nvt.xtc` to trajectory processing and
   phase-coexistence analysis.

The manuscript benchmark settings are 300 ns NPT at 0.01 ps followed by
3 μs NVT at 0.02 ps. A short full-workflow mode validates all transitions
without claiming a physically equilibrated phase-separated state.


In [ ]:
#@title Install DROPPS 1.0
#@markdown The default route installs the immutable wheel attached to the
#@markdown public GitHub `v1.0.0` release. Upload remains available for offline use.
installation_source = "Install from the DROPPS v1.0.0 GitHub release" #@param ["Install from the DROPPS v1.0.0 GitHub release", "Upload the DROPPS 1.0 wheel", "Install from another wheel URL"]
wheel_url = "https://github.com/tymworld/DROPPS/releases/download/v1.0.0/dropps-1.0-py3-none-any.whl" #@param {type:"string"}
accelerator_dependencies = "auto" #@param ["auto", "cuda12", "cuda13", "latest"]

import importlib.metadata
import re
import shutil
import subprocess
import sys
from pathlib import Path


def detect_install_extra():
    if accelerator_dependencies != "auto":
        return accelerator_dependencies
    if shutil.which("nvidia-smi"):
        probe = subprocess.run(
            ["nvidia-smi"], text=True, capture_output=True, check=False
        ).stdout
        match = re.search(r"CUDA Version:\s*(\d+)", probe)
        if match and int(match.group(1)) >= 13:
            return "cuda13"
        return "cuda12"
    return "latest"


if installation_source.startswith("Upload"):
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "This upload form is intended for Google Colab. Set installation_source "
            "to the URL option when running elsewhere."
        ) from exc
    uploaded = files.upload()
    wheel_candidates = [Path(name) for name in uploaded if name.endswith(".whl")]
    if len(wheel_candidates) != 1:
        raise ValueError("Upload exactly one DROPPS 1.0 .whl file.")
    wheel_target = str(wheel_candidates[0].resolve())
else:
    if not wheel_url.strip():
        raise ValueError("Provide the published DROPPS 1.0 wheel URL.")
    wheel_target = wheel_url.strip()

extra = detect_install_extra()
if wheel_target.startswith(("https://", "http://")):
    package_spec = f"dropps[{extra}] @ {wheel_target}"
else:
    package_spec = f"{wheel_target}[{extra}]"
print(f"Installing {package_spec}")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        package_spec,
    ],
    check=True,
)

version = importlib.metadata.version("dropps")
if version != "1.0":
    raise RuntimeError(f"Expected DROPPS 1.0, but installed {version}.")
subprocess.run(["dps", "--version"], check=True)


In [ ]:
#@title Shared notebook helpers
import hashlib
import json
import os
import shlex
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

COMMAND_LOG = []


def run_command(arguments, *, cwd=None, input_text=None, check=True):
    command = [str(value) for value in arguments]
    print("$", shlex.join(command))
    result = subprocess.run(
        command,
        cwd=None if cwd is None else str(cwd),
        input=input_text,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    COMMAND_LOG.append(
        {
            "time_utc": datetime.now(timezone.utc).isoformat(),
            "cwd": str(Path(cwd or Path.cwd()).resolve()),
            "command": command,
            "returncode": result.returncode,
        }
    )
    if check and result.returncode:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {shlex.join(command)}"
        )
    return result


def dps(*arguments, cwd=None, input_text=None, check=True):
    return run_command(
        ["dps", *arguments], cwd=cwd, input_text=input_text, check=check
    )


def reset_task_directory(path):
    path = Path(path).resolve()
    if not path.name.startswith("dropps_"):
        raise ValueError(f"Refusing to reset unexpected directory: {path}")
    if path.exists():
        import shutil

        shutil.rmtree(path)
    path.mkdir(parents=True)
    return path


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_extract_zip(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        bundle.extractall(destination)


def find_unique(root, basename):
    matches = [path for path in Path(root).rglob(basename) if path.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one file named {basename!r} below {root}, found {len(matches)}."
        )
    return matches[0]


def make_zip(paths, output_path, *, base=None):
    output_path = Path(output_path)
    base = Path(base or output_path.parent).resolve()
    with zipfile.ZipFile(output_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for source in sorted({Path(path).resolve() for path in paths}):
            if source == output_path.resolve() or not source.is_file():
                continue
            try:
                arcname = source.relative_to(base)
            except ValueError:
                arcname = Path(source.name)
            bundle.write(source, arcname)
    print(f"Wrote {output_path} ({output_path.stat().st_size / 1024:.1f} KiB)")
    return output_path


def download(path):
    try:
        from google.colab import files
    except ImportError:
        print(f"Result available at {Path(path).resolve()}")
    else:
        files.download(str(path))


def optional_time_arguments(start, end, interval):
    arguments = []
    for option, value in (("-b", start), ("-e", end), ("-dt", interval)):
        if str(value).strip():
            arguments.extend([option, str(value).strip()])
    return arguments


def read_xvg(path):
    legends = {}
    metadata = {}
    rows = []
    with Path(path).open(encoding="utf-8") as stream:
        for raw in stream:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("@"):
                re_module = __import__("re")
                match = re_module.search(r's(\d+)\s+legend\s+"(.*)"', line)
                if match:
                    legends[int(match.group(1)) + 1] = match.group(2)
                for key, pattern in {
                    "title": r'@\s+title\s+"(.*)"',
                    "xlabel": r'@\s+xaxis\s+label\s+"(.*)"',
                    "ylabel": r'@\s+yaxis\s+label\s+"(.*)"',
                }.items():
                    label_match = re_module.search(pattern, line)
                    if label_match:
                        metadata[key] = label_match.group(1)
                continue
            rows.append([float(value) for value in line.split()])
    data = np.asarray(rows, dtype=float)
    if data.ndim != 2 or data.shape[1] < 2:
        raise ValueError(f"No plottable data in {path}")
    return data, legends, metadata


def plot_xvg(path, *, title=None, output=None):
    data, legends, metadata = read_xvg(path)
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    for column in range(1, data.shape[1]):
        ax.plot(data[:, 0], data[:, column], label=legends.get(column, f"series {column}"))
    ax.set_title(title or metadata.get("title", Path(path).stem))
    ax.set_xlabel(metadata.get("xlabel", "x / time"))
    ax.set_ylabel(metadata.get("ylabel", "value"))
    if data.shape[1] > 2:
        ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    output_path = Path(output) if output is not None else Path(path).with_suffix(".png")
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    print(f"Saved figure: {output_path}")
    plt.show()
    return fig, output_path


In [ ]:
#@title Upload or reuse a workflow bundle
input_source = "Upload ZIP or individual files" #@param ["Upload ZIP or individual files", "Reuse files from an earlier notebook in this runtime"]
reuse_directory = "/content/dropps_model_builder" #@param {type:"string"}

import shutil

BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
INPUT_ROOT = reset_task_directory(BASE / "dropps_uploaded_inputs")

if input_source.startswith("Upload"):
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Upload this bundle in Colab or choose the reuse option.") from exc
    uploaded = files.upload()
    for name in uploaded:
        source = Path(name)
        if source.suffix.lower() == ".zip":
            safe_extract_zip(source, INPUT_ROOT)
        else:
            shutil.copy2(source, INPUT_ROOT / source.name)
else:
    source_root = Path(reuse_directory)
    if not source_root.is_dir():
        raise FileNotFoundError(source_root)
    for source in source_root.rglob("*"):
        if source.is_file() and source.suffix.lower() in {
            ".pdb", ".itp", ".top", ".mdp", ".tpr", ".xtc", ".ndx", ".chk", ".xml", ".json"
        }:
            target = INPUT_ROOT / source.name
            if target.exists() and sha256(target) != sha256(source):
                raise ValueError(f"Duplicate filename with different content: {source.name}")
            shutil.copy2(source, target)

print("Available files:")
for path in sorted(INPUT_ROOT.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(INPUT_ROOT), f"({path.stat().st_size / 1024:.1f} KiB)")


## 1 Resolve the compact cubic input system


In [ ]:
#@title Input filenames
structure_filename = "system.pdb" #@param {type:"string"}
topology_filename = "system.top" #@param {type:"string"}
require_cubic_input = True #@param {type:"boolean"}
cubic_relative_tolerance = 0.001 #@param {type:"number"}

BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
WORKDIR = reset_task_directory(BASE / "dropps_simulation")
INPUTS = WORKDIR / "inputs"
INPUTS.mkdir()

import shutil

for source in INPUT_ROOT.rglob("*"):
    if source.is_file() and (
        source.suffix.lower() in {".pdb", ".itp", ".top", ".ff", ".chk", ".xml", ".json", ".mdp", ".tpr"}
        or source.name.endswith(".state.xml")
        or source.name.endswith(".restart.json")
    ):
        target = INPUTS / source.name
        if target.exists() and sha256(target) != sha256(source):
            raise ValueError(f"Conflicting duplicate input: {source.name}")
        shutil.copy2(source, target)

STRUCTURE = find_unique(INPUTS, structure_filename)
TOPOLOGY = find_unique(INPUTS, topology_filename)

def pdb_box_lengths_nm(path):
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if line.startswith("CRYST1"):
            return np.asarray(
                [float(line[6:15]), float(line[15:24]), float(line[24:33])]
            ) / 10.0
    raise ValueError(f"No CRYST1 box record found in {path}")

compact_box_nm = pdb_box_lengths_nm(STRUCTURE)
relative_spread = np.ptp(compact_box_nm) / compact_box_nm.mean()
if require_cubic_input and relative_spread > cubic_relative_tolerance:
    raise ValueError(
        "The slab protocol must start from a compact cubic box; "
        f"received {compact_box_nm.tolist()} nm."
    )
print("Structure:", STRUCTURE)
print("Topology:", TOPOLOGY)
print("Compact box (nm):", compact_box_nm)


## 2 Define the NPT → z×10 → NVT protocol


In [ ]:
#@title Stage durations, integration, and temperature
workflow_action = "Short full-workflow validation" #@param ["Short full-workflow validation", "Run full configured workflow", "Prepare NPT TPR only", "Prepare NVT TPR from uploaded npt.pdb", "Run NVT from uploaded npt.pdb"]
production_temperature_K = 300.0 #@param {type:"number"}
npt_time_step_ps = 0.01 #@param {type:"number"}
npt_duration_ns = 300.0 #@param {type:"number"}
nvt_time_step_ps = 0.02 #@param {type:"number"}
nvt_duration_ns = 3000.0 #@param {type:"number"}
validation_steps_per_stage = 100 #@param {type:"integer"}
friction_per_ps = 1.0 #@param {type:"number"}
random_seed = 1215 #@param {type:"integer"}
minimize_before_npt = True #@param {type:"boolean"}
minimization_force_tolerance = 100.0 #@param {type:"number"}
minimization_max_steps = 100000 #@param {type:"integer"}
slab_axis = "z" #@param ["z", "x", "y"]
slab_box_multiplier = 10.0 #@param {type:"number"}

if slab_axis != "z":
    raise ValueError("This Colab phase-coexistence workflow expands the z axis.")
if slab_box_multiplier != 10.0:
    raise ValueError("This phase-coexistence workflow requires a 10-fold box expansion.")
if min(npt_time_step_ps, nvt_time_step_ps, npt_duration_ns, nvt_duration_ns) <= 0:
    raise ValueError("Time steps and stage durations must be positive.")

configured_npt_steps = int(round(npt_duration_ns * 1000 / npt_time_step_ps))
configured_nvt_steps = int(round(nvt_duration_ns * 1000 / nvt_time_step_ps))
short_workflow = workflow_action == "Short full-workflow validation"
npt_steps = validation_steps_per_stage if short_workflow else configured_npt_steps
nvt_steps = validation_steps_per_stage if short_workflow else configured_nvt_steps

print(f"NPT: {npt_steps:,} steps at {npt_time_step_ps:g} ps")
print(f"NVT: {nvt_steps:,} steps at {nvt_time_step_ps:g} ps")
print(f"Slab conversion: {slab_axis} axis × {slab_box_multiplier:g}")


In [ ]:
#@title Bond, COM, and nonbonded settings
bond_treatment = "bond" #@param ["bond", "constraint"]
center_of_mass_mode = "Linear" #@param ["Linear", "none"]
center_of_mass_interval = 100 #@param {type:"integer"}
vdw_type = "pLJ" #@param ["pLJ", "MPiPi"]
lj_cutoff_scheme = "static" #@param ["static", "dynamic"]
lj_cutoff_nm = 1.5 #@param {type:"number"}
lj_sigma_multiplier = 3.0 #@param {type:"number"}
shift_lj = True #@param {type:"boolean"}
coulomb_type = "yukawa" #@param ["yukawa", "no"]
coulomb_cutoff_nm = 1.5 #@param {type:"number"}
shift_coulomb = True #@param {type:"boolean"}
salt_concentration_molar = 0.1 #@param {type:"number"}


In [ ]:
#@title Pressure and output settings
reference_pressure_bar = 1.0 #@param {type:"number"}
barostat_attempt_interval = 5 #@param {type:"integer"}
trajectory_interval_ns = 1.0 #@param {type:"number"}
screen_log_interval_ns = 10.0 #@param {type:"number"}
file_log_interval_ns = 1.0 #@param {type:"number"}
energy_interval_ns = 1.0 #@param {type:"number"}
energy_groups = "potentialEnergy,kineticEnergy,totalEnergy,temperature,boxX,boxY,boxZ,volume,density" #@param {type:"string"}
stress_interval_steps = 0 #@param {type:"integer"}
stress_strain = 0.0003 #@param {type:"number"}
stress_platform = "auto" #@param ["auto", "CUDA", "OpenCL", "CPU", "Reference"]
stress_precision = "double" #@param ["single", "mixed", "double"]


In [ ]:
#@title MDP renderer for both ensembles
def bool_text(value):
    return "True" if value else "False"

def interval_steps(interval_ns, dt_ps, total_steps):
    requested = max(1, int(round(interval_ns * 1000 / dt_ps)))
    if short_workflow:
        return min(requested, max(1, total_steps // 10))
    return requested

def render_mdp(*, stage, dt_ps, nsteps, pressure_coupling, minimize):
    effective_max_steps = (
        min(minimization_max_steps, 100)
        if short_workflow and minimize
        else minimization_max_steps
    )
    return f"""# DROPPS 1.0 Colab: {stage} stage of NPT -> slab -> NVT
integrator = Langevin
dt = {dt_ps}
nsteps = {nsteps}
friction = {friction_per_ps}
seed = {random_seed}
minimize = {bool_text(minimize)}
forcetol = {minimization_force_tolerance}
max-step = {effective_max_steps}
production-temperature = {production_temperature_K}
gen-vel = True
initial-temperature = {production_temperature_K}
warming-speed = 1.0
keep-warming-trajectory = False
bondtype = {bond_treatment}
comm-mode = {center_of_mass_mode}
nstcomm = {center_of_mass_interval}
vdwtype = {vdw_type}
cutoff-scheme-lj = {lj_cutoff_scheme}
cutoff-lj = {lj_cutoff_nm}
cutoff-lj-multi = {lj_sigma_multiplier}
shift-lj = {bool_text(shift_lj)}
coulombtype = {coulomb_type}
cutoff-coul = {coulomb_cutoff_nm}
shift-coul = {bool_text(shift_coulomb)}
salt-conc = {salt_concentration_molar}
pcoulp = {bool_text(pressure_coupling)}
ref-P = {reference_pressure_bar}
tau-P = {barostat_attempt_interval}
nst-xout = {interval_steps(trajectory_interval_ns, dt_ps, nsteps)}
nst-screenlog = {interval_steps(screen_log_interval_ns, dt_ps, nsteps)}
nst-filelog = {interval_steps(file_log_interval_ns, dt_ps, nsteps)}
nst-energy = {interval_steps(energy_interval_ns, dt_ps, nsteps)}
energy-grps = {energy_groups}
nst-stress = {stress_interval_steps}
stress-strain = {stress_strain}
stress-output = auto
stress-platform = {stress_platform}
stress-precision = {stress_precision}
stress-device = auto
stress-threads = 0
"""


## 3 Runtime, checkpoint, and restart controls


In [ ]:
#@title Runtime controls shared by both stages
compute_platform = "auto" #@param ["auto", "CUDA", "OpenCL", "CPU", "Reference"]
gpu_device_index = "0" #@param {type:"string"}
gpu_precision = "mixed" #@param ["single", "mixed", "double"]
cpu_threads = 0 #@param {type:"integer"}
checkpoint_minutes = 5.0 #@param {type:"number"}
keep_numbered_checkpoints = False #@param {type:"boolean"}
maximum_wall_hours = -1.0 #@param {type:"number"}
restart_npt_from_checkpoint = False #@param {type:"boolean"}
npt_checkpoint_filename = "" #@param {type:"string"}
restart_nvt_from_checkpoint = False #@param {type:"boolean"}
nvt_checkpoint_filename = "" #@param {type:"string"}
append_restart_outputs = True #@param {type:"boolean"}

def run_stage(tpr, prefix, restart=False, checkpoint_filename=""):
    arguments = [
        "mdrun", "-s", Path(tpr).name, "-o", prefix,
        "--platform", compute_platform, "-cpt", checkpoint_minutes,
        "-maxh", maximum_wall_hours,
    ]
    if compute_platform in {"CUDA", "OpenCL"}:
        arguments.extend(["--precision", gpu_precision])
        if gpu_device_index.strip():
            arguments.extend(["-gpu_id", gpu_device_index.strip()])
    elif compute_platform == "CPU" and cpu_threads > 0:
        arguments.extend(["-nt", cpu_threads])
    if keep_numbered_checkpoints:
        arguments.append("-cpnum")
    if restart:
        arguments.append("-cpi")
        if checkpoint_filename.strip():
            arguments.append(checkpoint_filename.strip())
    arguments.append("-append" if append_restart_outputs else "-noappend")
    dps(*arguments, cwd=INPUTS)


## 4 Stage 1 — compact cubic-box NPT

Pressure coupling is mandatory in this stage. The full defaults generate the
condensed configuration over 300 ns. `Prepare NPT TPR only` is intended for
transfer to a persistent GPU/HPC resource.


In [ ]:
#@title Compile and optionally execute NPT
NPT_MDP = INPUTS / "npt.mdp"
NPT_TPR = INPUTS / "npt.tpr"
run_npt_here = workflow_action in {
    "Short full-workflow validation", "Run full configured workflow"
}
use_existing_npt = workflow_action in {
    "Prepare NVT TPR from uploaded npt.pdb", "Run NVT from uploaded npt.pdb"
}

if not use_existing_npt:
    NPT_MDP.write_text(
        render_mdp(
            stage="compact NPT", dt_ps=npt_time_step_ps,
            nsteps=npt_steps, pressure_coupling=True,
            minimize=minimize_before_npt,
        ),
        encoding="utf-8",
    )
    dps(
        "grompp", "-f", STRUCTURE.name, "-p", TOPOLOGY.name,
        "-m", NPT_MDP.name, "-o", NPT_TPR.name, cwd=INPUTS,
    )
    dps("check", "-s", NPT_TPR.name, cwd=INPUTS)

if run_npt_here:
    run_stage(
        NPT_TPR, "npt", restart_npt_from_checkpoint,
        npt_checkpoint_filename,
    )
    dps("check", "-s", NPT_TPR.name, "-f", "npt.xtc", cwd=INPUTS)
    NPT_FINAL = INPUTS / "npt.pdb"
elif use_existing_npt:
    NPT_FINAL = find_unique(INPUTS, "npt.pdb")
else:
    NPT_FINAL = None
    print("NPT TPR prepared. Run it to completion before constructing the slab.")


## 5 Stage 2 — tenfold box expansion and elongated-box NVT

`dps editconf -mz 10` unwraps the expanded axis, increases only that box
length, and translates the NPT configuration to the center of the elongated
box. The generated `slab_nvt.mdp` has `pcoulp = False`. Because a PDB does
not store velocities, the NVT TPR generates new velocities at the same target
temperature.


In [ ]:
#@title Build and optionally execute the NVT slab stage
SLAB_PDB = INPUTS / "slab.pdb"
NVT_MDP = INPUTS / "slab_nvt.mdp"
NVT_TPR = INPUTS / "slab_nvt.tpr"
run_nvt_here = workflow_action in {
    "Short full-workflow validation", "Run full configured workflow",
    "Run NVT from uploaded npt.pdb",
}

if NPT_FINAL is not None:
    npt_box_nm = pdb_box_lengths_nm(NPT_FINAL)
    multiplier_option = {"x": "-mx", "y": "-my", "z": "-mz"}[slab_axis]
    dps(
        "editconf", "-f", NPT_FINAL.name, "-o", SLAB_PDB.name,
        multiplier_option, slab_box_multiplier, cwd=INPUTS,
    )
    slab_box_nm = pdb_box_lengths_nm(SLAB_PDB)
    axis_index = {"x": 0, "y": 1, "z": 2}[slab_axis]
    ratios = slab_box_nm / npt_box_nm
    expected = np.ones(3)
    expected[axis_index] = slab_box_multiplier
    if not np.allclose(ratios, expected, rtol=2e-3, atol=2e-3):
        raise RuntimeError(
            f"Slab-box verification failed: ratios={ratios.tolist()}, "
            f"expected={expected.tolist()}"
        )
    print("NPT box (nm):", npt_box_nm)
    print("Slab box (nm):", slab_box_nm)
    print("Verified box-length ratios:", ratios)

    NVT_MDP.write_text(
        render_mdp(
            stage="elongated-box NVT", dt_ps=nvt_time_step_ps,
            nsteps=nvt_steps, pressure_coupling=False, minimize=False,
        ),
        encoding="utf-8",
    )
    dps(
        "grompp", "-f", SLAB_PDB.name, "-p", TOPOLOGY.name,
        "-m", NVT_MDP.name, "-o", NVT_TPR.name, cwd=INPUTS,
    )
    dps("check", "-s", NVT_TPR.name, cwd=INPUTS)

    if run_nvt_here:
        run_stage(
            NVT_TPR, "slab_nvt", restart_nvt_from_checkpoint,
            nvt_checkpoint_filename,
        )
        dps(
            "check", "-s", NVT_TPR.name, "-f", "slab_nvt.xtc",
            cwd=INPUTS,
        )
        print("Analysis input: slab_nvt.tpr + slab_nvt.xtc")
    else:
        print("NVT TPR prepared: slab_nvt.tpr")


## 6 Export the staged simulation and explicit analysis handoff


In [ ]:
#@title Save manifest and download simulation bundle
files_to_export = [path for path in INPUTS.iterdir() if path.is_file()]
analysis_ready = (INPUTS / "slab_nvt.tpr").is_file() and (INPUTS / "slab_nvt.xtc").is_file()
manifest = {
    "notebook": "02_DROPPS_1_0_Simulation.ipynb",
    "dropps_version": "1.0",
    "workflow_action": workflow_action,
    "protocol": [
        "compact cubic box",
        "NPT condensation/equilibration",
        f"{slab_axis}-axis x{slab_box_multiplier:g}",
        "elongated-box NVT production",
        "analyse NVT trajectory only",
    ],
    "scientific_parameters": {
        "temperature_K": production_temperature_K,
        "npt": {
            "dt_ps": npt_time_step_ps,
            "configured_duration_ns": npt_duration_ns,
            "effective_steps": npt_steps,
            "pressure_coupling": True,
        },
        "slab_expansion": {"axis": slab_axis, "multiplier": slab_box_multiplier},
        "nvt": {
            "dt_ps": nvt_time_step_ps,
            "configured_duration_ns": nvt_duration_ns,
            "effective_steps": nvt_steps,
            "pressure_coupling": False,
        },
        "forcefield_vdw_type": vdw_type,
        "salt_concentration_molar": salt_concentration_molar,
        "seed": random_seed,
    },
    "operational_parameters": {
        "platform": compute_platform,
        "precision": gpu_precision,
        "checkpoint_minutes": checkpoint_minutes,
        "maximum_wall_hours": maximum_wall_hours,
    },
    "analysis_handoff": {
        "ready": analysis_ready,
        "run_input": "slab_nvt.tpr",
        "trajectory": "slab_nvt.xtc",
        "ensemble": "NVT",
    },
    "files": {path.name: sha256(path) for path in files_to_export},
    "commands": COMMAND_LOG,
}
manifest_path = INPUTS / "simulation_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
files_to_export.append(manifest_path)
archive = make_zip(
    files_to_export,
    WORKDIR / "DROPPS_1_0_simulation_bundle.zip",
    base=INPUTS,
)
download(archive)
